Tasks:
1. Add links to different notebooks

# Fermionic Gaussian States

## Overview

## Introduction

Fermionic Gaussian States (FGS) are a class of many-body quantum states of fermions, which are completely characterized by their "second moments" (two-point correlation functions). They underpin many approximate techniques in condensed matter physics and quantum chemistry. In these approaches, the interacting many-electron problem are treated by determining the optimal set of effective non-interacting orbitals that best approximate the full system [[1](#numerical)].
The structure FGSs are governed by the anticommutation relations of the Dirac operators

$$\{c_i, c_j^\dagger \} = \delta_{ij}~~,~~\{c_i, c_j \} = \{c_i^\dagger, c_j^\dagger \} =0~~,$$ 

where $\{a,b\} = ab + ba$, while $c_i$ and $c_i^\dagger$ annihilate and create a fermion in the $i$'th fermionic mode, respectively.


Important fermionic gaussian states include,
- The vaccum state $|\text{vacc}\rangle$.
- Slater Determinants: $\Pi_{k=1}^M b_k^\dagger |\text{vacc}\rangle$, where $b_k$ are linear combinations of $\{c_i\}$. These include Hartree-Fock states and a Filled Fermi sea.
- BCS  states: $$\propto \exp \left( \sum_{ij}\Delta_{ij}c_i^\dagger c_j^\dagger \right)|\text{vacc}\rangle~~. $$ These states, named after novel literates Bardeen, Cooper and Schrieffer, describe, the pairing of electrons (forming Cooper pairs), which leads to superconducting charge flow in solid state materials.
- Ground states of quadratic Hamiltonians: For an arbitrary quadratic fermionic Hamiltonian
 $$ H = \sum_{\mu \nu} c_\mu^\dagger h_{\mu \nu} c_{\nu} + \frac{1}{2}\sum_{\mu nu}\Delta_{\mu \nu}\left(c_\mu^\dagger c_\nu^\dagger +\text{h.c} \right)~~,~~~~(1)$$
  where h.c denotes the hermitian conjugate, $h$ is hermitian ($h=h^\dagger$) and $\Delta$ is skew-symmetric ($\Delta = -\Delta^T$). The greek indices represent both the spatial and spin degrees of freedom. The ground state is a fermionic gaussian state.
As ground states of such Hamiltonians, they have an important role in quantum simulations of fermionic systems.
For example, in the celebrated Fermi-Hubbard model ADD LINK TO THE NOTEBOOK, simulations are usually initialized in easy to prepare fermionic gaussian state, or as ground states of mean-field Hamiltonians.

COMPLETE THE FOLLOWING

They have a number of equivalent properties:
Consider an $N$-particle fermionic gaussian state $\rho$
- $\rho$ can be represented as the exponential of a quadratic form in the fermionic operators: $$\propto \exp\left(-\frac{i}{4} \gamma^T A \gamma \right)~~,$$ where $A$ is a real skew-symmetric $2N\times 2N$ matrix.
- All  higher order correlation functions (i.e. $\langle c_1^\dagger...c_M^\dagger c_{M+1}\dots c_{2M} \rangle$) are determined by Wick's theorem from the two-point correlators.
- Alternatively, their properties can be fully described by the convenience matrix $\Gamma$, with elements $$\Gamma_{\mu \nu} = \text{tr}\left(\rho  \right) ~~.$$

## Classiq Implementation

We begin by importing the required software packages and defining global constants

In [1]:
# Uncomment to install openfermion python package
#!pip install openfermion

In [3]:
import numpy as np
import pandas as pd
from openfermion.circuits.slater_determinants import (
    slater_determinant_preparation_circuit,
)
from openfermion.linalg.givens_rotations import (
    fermionic_gaussian_decomposition,
    givens_decomposition,
)
from openfermion.ops import QuadraticHamiltonian

from classiq import *

ModuleNotFoundError: No module named 'pandas'

### Preparation of a Slater Determinant State

A Slater-Determinant is the ground state of a quadratic Fermionic Hamiltonian which conserves the number of particles.

For a general number conserving quadratic fermionic Hamiltonian ($\Delta = 0$ in Eq. (1)) $$H = \sum_{\mu\nu}c_\mu^\dagger h_{\mu\nu}c_\nu~~,~~~~(2)$$ where $h$ is known as the **one-body Hamiltonian**.

Using the anti-commutation relations and the Heisenberg equation ($\frac{dO(t)}{dt} = i\left[H, O(t) \right]$), the dynamics in the Heisenberg representation can be expressed as $$\mathbf{c}^\dagger (t) = e^{i H t} \mathbf{c}^\dagger e^{-i H t}  = e^{-i h^T }\mathbf{c}^\dagger~~,$$ where $\mathbf{c}^\dagger = \{c_1^\dagger,\dots,c_M^\dagger\}^T$ and $M$ is the total number of fermionic modes. Remarkably, this implies that the diagonalization of $h$, ($M$ by $M$ matrix) provides the dynamics in the $2^M$ Hilbert space.


Diagonalizing the single particle Hamiltonian $M= \bar{Q} D \bar{Q}^\dagger$, where $\bar{Q}$ is and $M$-by-$M$ unitary matrix and $D = \text{diag}(\epsilon_1,...,\epsilon_M)$, we obtain $H=\sum_{k=1}^M \epsilon_k d_k^\dagger d_k$, where $$d_\eta^\dagger= \sum_{\mu=1}^{M} \bar{Q}_{\mu \eta}c_\mu^\dagger~~. \tag{4}$$
Alternatively, the basis transformation can be expressed in terms of the single-particle transformation, $U\mathbf{c}^\dagger = \mathbf{d}^\dagger~,$ where we collected the creation operators to form an operator valued vector: $\mathbf{c}^\dagger = \{c_1^\dagger,\dots,c_M^\dagger\}^T$ and similarly for $\mathbf{d}^\dagger$.

For a fixed particle number $N$ the ground state is given by
$$|\psi_{g.s}\rangle =\Pi_{k} {\cal U}c_k^\dagger |0^M\rangle =  \Pi_{k=1}^M d_k^\dagger |0^M\rangle ~~,$$
where $i$ iterates over the occupied fermionic modes and ${\cal U} c_j^\dagger {\cal U}^\dagger = U_{[j,:]} \mathbf{d}^\dagger =  d_j^\dagger$.
The diagonalization of $h$ scales only polynomially with the number of lattice sites, $O(L^3)$ and can be efficiently done on a classical computer.

In order to prepare the desired ground state we focus on a part of $\bar{Q}$ which corresponds to the occupied modes, and denote the  $N$ by $M$ matrix describing these modes by $Q = (\bar{Q}^T)_{[{\text{occupied modes}},:]}$. This identification leads to an alternative form for Eq. (4), for the occupied modes we have $$d_\eta^\dagger= \sum_{\mu=1}^{M} {Q}_{\eta \mu}c_\mu^\dagger~~.$$

An efficient quantum circuit for the ground state preparation can be obtained by a modified QR decomposition of $Q$, using a product of elementary two-mode rotations, called Given rotations. Each rotation can be expressed as
$$
\begin{pmatrix}
\mathcal{G}c_k^\dagger\mathcal{G}\\
\mathcal{G}c_j^\dagger\mathcal{G}
\end{pmatrix}
=
G(\theta,\varphi)\,
\begin{pmatrix}
c_k\\
c_j
\end{pmatrix}~~,
 $$
and the form of a Given rotation is  $$G_{jk}(\theta,\varphi) = \begin{pmatrix}
\cos(\theta) & -e^{i\varphi}\sin(\theta)\\
\sin(\theta) & e^{i\varphi}\cos(\theta)
\end{pmatrix}~~. $$

To simplify the decomposition we begin by utilizing the invariance of the Slater determinant (up to a global phase) under the mapping ${Q}\rightarrow V{Q}$, where $V$ is a unitary transformation. For the transformation of basis to be valid we require that the first $N$ rows of $VQ$ are equal to $U$, or alternatively $$V{Q}U^\dagger = (I_N, \boldsymbol{0})~~.$$ Each Given transformation operates only on two columns and it's parameters, $\theta$ and $\phi$ are set so to nullify elements in the upper right part of the matrix. Due to the orthogonality of the rows of $Q$, some transformations nullify more than a single matrix element. As a result, the total number of required Given transformations are $N_G = N(M-N)$, where $N$ is the number of electrons and $M$ is the number of fermionic modes. The diagonalization procedure results in a product of Given rotations $$U = G_{N_G}\cdots G_2 G_1~~.$$

After the Jordan-Wigner transformation, each two-mode ($j,k$) rotations correspond to a rotation in the single particle subspace of the two qubits $j$ and $k$ ($|01\rangle$ and $|10\rangle$). This completely, defines the state preparation circuit in terms of a sequence two-qubit rotations.
The gate complexity is $O(N_G) = O(N^2)$ (worst case achieved for $M=N/2$), and parallelization leads to a circuit depth of $M-1$.

### Summary of the Algorithmic Steps:
1. Evaluate the diagonalizing matrix $\bar{Q}$, and filter the non-occupied states by calculating $Q$.
2. Zero out the upper-right matrix elements of $Q$, applying the transformation $Q\rightarrow V Q$.
3. Diagonalize $VQ$ applying a sequence of given rotations.
4. Map the Given rotations to corresponding quantum gates, to obtain the quantum circuit.


#### Given Rotations Example
In order to understand the state preparation algorithm, we first show how Given rotations are utilized to diagonalize a simple one-body Hamiltonian.

Consider a simple $4$-by-$4$ Hermitian matrix, representing a one-body Hamiltonian:
$$h =
\begin{pmatrix}
\varepsilon_0 & t_{01} & 0 & t_{03} \\
t_{01} & \varepsilon_1 & t_{12} & 0 \\
0 & t_{12} & \varepsilon_2 & t_{23} \\
t_{03} & 0 & t_{23} & \varepsilon_3
\end{pmatrix}
 $$

In [ ]:
## Defining the single-body Hamiltonian
# Onsite energies
eps0, eps1, eps2, eps3 = 0.8, -0.4, 0.3, -0.1

# Hopping amplitudes (real for simplicity)
t01 = 0.25
t12 = -0.35
t23 = 0.20
t03 = 0.15

# 4x4 Hermitian one-body matrix h_{pq}
h = np.array(
    [
        [eps0, t01, 0.0, t03],
        [t01, eps1, t12, 0.0],
        [0.0, t12, eps2, t23],
        [t03, 0.0, t23, eps3],
    ],
    dtype=complex,
)

We employ the methods of OpenFermion's `QuadraticHamiltonian` class in order to extract the Given rotations.
The number conserving quadratic Hamiltonian is then diagonalized utilizing a Bogoliubov transformation, leading to the `orbital_energies` and a `transformation_matrix`.

In [ ]:
# Number-conserving quadratic Hamiltonian (no pairing term)
qh = QuadraticHamiltonian(h, constant=0.0)
orbital_energies, transformation_matrix, constant = (
    qh.diagonalizing_bogoliubov_transform()
)

# Taking the number of electrons to be equal to be two
occupied_orbitals = np.where(orbital_energies < 0.0)[0]
slater_determinant_matrix = transformation_matrix[occupied_orbitals]

E, Qbar = np.linalg.eigh(h)
assert np.allclose(transformation_matrix, Qbar.T)

We extract the given rotations utilizing OpenFermion's `given_decomposition`, returning a list of tuples: `[(G_1,G_2),(G_3,),...]`. Each tuple includes the Given rotations which can be operated in parallel. The Given rotations are encoded as a tuple: $G_k = (i_k,j_k,\theta_k,\phi_k)$, where $i_k$ and $i_k$ are the columns of $U^\dagger$ which the Given rotation $G_k^\dagger$ is operated on from the right (see calculation below).

In [ ]:
rotations, V, diag = givens_decomposition(slater_determinant_matrix)

We introduce two utility functions to demonstrate the decomposition, `given_gate` implements the two-by-two matrix and `build_U_from_rotations`, gathers all the Given rotations to create $U$.

In [ ]:
def givens_gate(theta: float, phi: float) -> np.ndarray:
    """
    Givens rotation:
     G(i,j,theta, phi) =    [[ cos(theta), -e^{i phi} sin(theta)],
                            [ sin(theta),  e^{i phi} cos(theta)]]
    """
    c = np.cos(theta)
    s = np.sin(theta)
    e = np.exp(1j * phi)
    return np.array([[c, -e * s], [s, e * c]], dtype=complex)


def build_U_from_rotations(M: int, rotations) -> np.ndarray:
    """
    Reconstruct U (M x M) from OpenFermion 'givens_rotations' list.

    OpenFermion applies these to columns during decomposition; updating columns
    by right-multiplying with G^\dagger reproduces the same effect.
    """
    U_dagger = np.eye(M, dtype=complex)
    for parallel_ops in rotations:
        for i, j, theta, phi in parallel_ops:
            G = givens_gate(theta, phi)
            cols = U_dagger[:, [i, j]]
            U_dagger[:, [i, j]] = cols @ G.conj().T
    return U_dagger.conj().T


def build_state_from_rotations(M: int, N: int, circuit_description: list) -> np.ndarray:
    v = np.zeros((M,), dtype=complex)
    v[:N] = np.ones_like(v[:N])
    for parallel_ops in circuit_description:
        for i, j, theta, phi in parallel_ops:
            G = givens_gate(theta, phi)
            v[[i, j]] = G @ v[[i, j]]
    return v

Next, we decompose $U$ into Given rotations: $$U = G_{N_G},\dots,G_1 $$ and verify that $V Q U^\dagger = (I,\mathbf{0})$.

In [ ]:
tol = 1e-8
Q = np.asarray(slater_determinant_matrix, dtype=complex)
n, m = Q.shape

U = build_U_from_rotations(m, rotations)

# Check V Q.T U^† = D  where D has diag entries in first m columns and zeros elsewhere
D = np.zeros((n, m), dtype=complex)
D[np.arange(n), np.arange(n)] = diag

A = V @ Q @ U.conj().T

# Normalize to (I,0) by removing the diagonal unitary on the left:
# Let Dm = diag(diag) (m x m). Then Dm^† (V Q U^†) = (I,0).
# So define V' = Dm^† V.
Dm_dag = np.diag(
    np.conjugate(diag)
)  # Dm^† since diag entries are unit-modulus in theory

Vprime = Dm_dag @ V

I0 = np.zeros((n, m), dtype=complex)
I0[:, :n] = np.eye(n, dtype=complex)

B = Vprime @ Q @ U.conj().T
assert np.allclose(A, D, atol=tol, rtol=tol)
assert np.allclose(B, I0, atol=tol, rtol=tol)

State preparation check

In [ ]:
# State preparation check for the single excitation subspace
slater_determinant_matrix = transformation_matrix[[0]]
rotations, V, diag = givens_decomposition(slater_determinant_matrix)
circuit_description = reversed(rotations)
ground_state = build_state_from_rotations(m, n, circuit_description)
assert np.allclose(h @ Qbar[:, 0], E[0] * Qbar[:, 0], atol=tol, rtol=tol)

Next, we introduce quantum functions that implement the state preparation circuit $U = G_{N_G}\dots G_1$.

In [ ]:
@qfunc
def G(theta: float, phi: float, qba: QArray[QBit, 2]):
    """
    Implements a Given rotation on two qubits.
    """
    c = np.cos(theta)
    s = np.sin(theta)
    e = np.exp(1j * phi)
    U = [
        [1, 0, 0, 0],
        [0, c, -e * s, 0],
        [0, s, e * c, 0],
        [0, 0, 0, 1],
    ]
    unitary(U, qba)


@qfunc
def given_rotation(gate: list[int, int, float, float], qba: QArray[QBit, M]) -> None:
    """
    Implements a Given rotation on two specific qubits, i and j, from an M-qubit quantum array, qba.
    """
    i, j, theta, phi = gate
    G(theta, phi, [qba[i], qba[j]])


@qfunc
def prepare_slater_det(h: list[list[float, M], M], N: int, qba: QArray[QBit, M]):
    """
    Prepares the ground state associated with the single electron matrix h.
    The Hamiltonian satisfies H = \sum_{\mu,\nu}c_{\mu}^\dagger h_{\mu,\nu} c_{\nu}

    Args:
        h (ndarray): single electron matrix
        N (int): number of electrons
        qba (list[QBit]): list of qubits
    """
    # preparing the reference state, as the state with the first N qubits in state |1> and the rest in state |0>.
    repeat(N, lambda i: X(qba[i]))

    # diagonalizing h
    D, Qbar = np.linalg.eigh(h)
    Q = (Qbar.T)[:N, :]  # occupying the N lowest energy states
    rotations, _, _ = givens_decomposition(Q)
    # U = build_U_from_rotations(M, rotations)
    circuit_description = list(reversed(rotations))
    # ground_state = build_state_from_rotations(M, N, circuit_description)
    for parallel_ops in circuit_description:
        for gate in parallel_ops:
            given_rotation(list(gate), qba)

The circuit is verified by considering the single excitation subspace and comparing the compared state to the analytical result.

The ground state of the single excitation subspace is up to a global phase just $$d_1^\dagger | 0^M\rangle =\sum_\mu Q_{1,\mu} c^{\dagger}_\mu | 0^M\rangle ~~,$$ which after the Jordan-Wigner transformation corresponds to the quibit state with amplitudes $$[Q_{1,1},Q_{1,2},\dots, Q_{1,M}]~~.$$

In order to prepare the initial state, we begin by constructing the initial Hamiltonian. Utility functions are defined and the Hamiltonian parameters are set.

In [ ]:
def sym(A):
    """
    Symmetrizes the matrix A
    """
    return (A + A.T) / 2


def kinetic_energy(L: int, J: float) -> np.ndarray:
    """
    Builds single electron matrix of nearest-neighbor hopping term, hopping strength t, associated with an L site periodic lattice.
    """
    K = np.zeros((2 * L, 2 * L), dtype=float)
    for site in range(L):
        for spin in range(2):
            mu = qubit_idx(site, spin)
            nu = qubit_idx(site + 1, spin)
            K[mu, nu] = -J
    K = 2 * sym(K)
    return K


def spin_potential(L: int, spin=0, parameters: tuple[float] = (1, 1, 1)) -> np.ndarray:
    """
    Builds single electron , matrix associated with the external potential. Associated with an L site periodic lattice.
    """
    lam, mean, std = parameters
    V = np.zeros((2 * L, 2 * L), dtype=float)
    for site in range(L):
        mu = qubit_idx(site, spin)
        V[mu, mu] = -lam * np.exp(-((site - mean) ** 2) / (2 * std**2))
    return V


def prepare_single_electron_hamiltonian(
    L: int, J: float, parameters: tuple
) -> np.ndarray:
    return kinetic_energy(L, J) + spin_potential(L, spin=0, parameters=parameters)

In [ ]:
lam = 10.0
mean = L / 2
std = L / 6
h = prepare_single_electron_hamiltonian(L, J=1, parameters=(lam, mean, std))
N = 1

In [ ]:
@qfunc
def main(qba: Output[QArray[QBit, M]]) -> None:
    allocate(M, qba)
    prepare_slater_det(h, N, qba)


qprog = synthesize(main)

The state preparation quantum circuit. The Given rotation, operating on two qubits, is decomposed into the basis gates.


ADD the CIRCUIT...

In [ ]:
# # single electron test
# execution_preferences = ExecutionPreferences(
#     backend_preferences=ClassiqBackendPreferences(
#         backend_name=ClassiqSimulatorBackendNames.SIMULATOR_STATEVECTOR
#     ),
# )
#
# with ExecutionSession(
#     quantum_program=qprog, execution_preferences=execution_preferences
# ) as es:
#     res = es.sample()
# # assert(np.isclose(classical_ground_state == result))

In [ ]:
backend_preferences = ClassiqBackendPreferences(
    backend_name=ClassiqSimulatorBackendNames.SIMULATOR_STATEVECTOR
)


# Construct a representation of HHL model
def state_check_model(main, backend_preferences):
    qmod_state_check = create_model(
        main,
        execution_preferences=ExecutionPreferences(
            num_shots=1, backend_preferences=backend_preferences
        ),
    )
    return qmod_state_check

In [ ]:
# Construct the quantum program
qmod_state_check = state_check_model(main, backend_preferences)
qprog_state_check = synthesize(qmod_state_check)
# show(qprog_state_check)

print("Circuit depth = ", qprog_state_check.transpiled_circuit.depth)
res_state_check = execute(qprog_state_check).result_value()

In [ ]:
import matplotlib.pyplot as plt


def get_quantum_amplitudes(result):
    df = result.dataframe
    mask = np.abs(df["amplitude"]) > 1e-12
    filtered = df.loc[mask, ["bitstring", "amplitude"]].copy()
    amps = filtered["amplitude"].to_numpy()
    bitstring = filtered["bitstring"].tolist()  # must be integers
    # mapping the bitstrings to qubit indices in the array
    qubit_index = [[i for i, b in enumerate(s[::-1]) if b == "1"] for s in bitstring]
    qubit_index = np.array([i[0] for i in qubit_index])
    idx = np.argsort(qubit_index)
    qubit_index = qubit_index[idx]
    amps = amps[idx]
    amps = amps / np.linalg.norm(amps)

    qsol = np.zeros(M, dtype=complex)
    qsol[qubit_index] = amps
    return qsol

For the single excitation subspace we can easily compare the preparation of the quantum state with the exact diagonalization. To compare between the two we normalize the quantum solution so to cancel the difference in the global phase with respect to the classical result.

In [ ]:
def quantum_solution_normalization(exact_amplitudes, result):
    qsol = get_quantum_amplitudes(result)
    global_phase = np.angle(np.vdot(qsol, exact_amplitudes))
    # Correct global phase and taking the real part
    qsol_corrected = np.real(np.exp(1j * global_phase) * qsol)

    return qsol_corrected


# Quantum amplitudes
quantum_amplitudes = get_quantum_amplitudes(res_state_check)

# Exact diagonalization
E0, V0 = np.linalg.eigh(h)
exact_ground_state = V0[:, 0] / np.linalg.norm(V0[:, 0])

quantum_amplitudes_corrected = quantum_solution_normalization(
    exact_ground_state, res_state_check
)
grid = np.arange(0, M)

plt.figure()
plt.title("Initial State")
plt.plot(grid, quantum_amplitudes_corrected, "o", label="quantum state")
plt.plot(
    grid,
    exact_ground_state,
    "+",
    markersize=16,
    label="exact state for single excitation state",
)
plt.xlim(0, M - 1)
plt.ylim(-1, 1)
plt.xlabel(r"Qubit Index")
plt.ylabel(r"Amplitudes")
plt.legend()
plt.show()

### General Quadratic Hamiltonian
In the general case the number of particles are not conserved under the dynamics of $H$ (Eq. (1)). In this case, the Hamiltonian can be written in a concise form $$ H = ~~,$$ for numerical diagonalization it is convenient to introduce the **Majorana operators**: $$x_j = \frac{1}{\sqrt{2}}\left(c_j^\dagger + c_j \right)~~~,~~~$$p_j = \frac{i}{\sqrt{2}}\left(c_j^\dagger - c_j \right)~~, $$ for $j=1,\dots ,M/2$$. The Majorana operators satisfy the  anticommutation relations $$\{x_i,x_j \}= \{p_i,p_j \} = \delta_{ij}~~,~~\{x_i, p_i\} = 0 ~~.$$ A unitary mapping relates between the two representations $$ ~~,$$ where $\Omega = $
Expressing the Hamiltonian in terms of the Majorana modes we have $ H = ~,$,

COMPLETE
Bring $A$ into standard form and obtain the basis transformation..

## References

<a id='numerical'>[1] </a>:Surace, J., & Tagliacozzo, L. (2022).Fermionic Gaussian states: an introduction to numerical approaches. SciPost Physics Lecture Notes, 054. [arXiv:2111.08343](https://arxiv.org/abs/2111.08343).


<a id='google_paper'>[2] </a>: Arute, F., Arya, K., Babbush, R., Bacon, D., Bardin, J. C., Barends, R., ... & Zanker, S. (2020). Observation of separated dynamics of charge and spin in the Fermi-Hubbard model. [arXiv:2010.07965](https://arxiv.org/abs/2010.07965).

<a id='fermionic_gaussian_state'>[3] Jiang, Z., Sung, K. J., Kechedzhi, K., Smelyanskiy, V. N., & Boixo, S. (2018). Quantum algorithms to simulate many-body physics of correlated fermions. [Physical Review Applied, 9(4), 044036](https://arxiv.org/abs/1711.05395).